# Product Recomendation Analysis

**Objectives**

Calculate Jaccard similarity scores for product co-purchases to generate 
product recommendation candidates.


**Key Outputs**

*Co-purchase count matrix* — pairwise count of products appearing together 
within the same invoice.

*Jaccard similarity matrix* — pairwise similarity scores normalised for 
product popularity, defined as:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

where $A$ and $B$ are the sets of invoices containing each product 
respectively. Jaccard similarity is preferred over raw co-purchase counts 
as it controls for frequently purchased products that co-occur by chance 
rather than by association.
**Data**

Order data is queried from `data/sqliteDB/sales.db` using `Q_pr` in 
`scripts/SQL_queries.py`, producing a table with the following fields:

| Field | Description |
|---|---|
| `InvoiceNo` | Unique invoice identifier |
| `CustomerID` | Unique customer identifier |
| `StockCode` | Unique product identifier |

The query incorporates a subquery `Qsub_Order_Data` which excludes records with `CategoryLabel` is `NULL`. These were identified during category creation and correspond to non-sales records such as broken stock and damages. Valid entries with no descriptions are retained and given 'NO DESCRIPTION' entry.


In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np 

In [2]:
# Query data
sys.path.insert(0, str(Path.cwd().parents[0])) # add project root to sys path
from scripts.SQL_queries import main as query

pr_df = query(pr=True)
pr_df = pr_df[['InvoiceNo','StockCode']]
pr_df.head()

,InvoiceNo,StockCode
0,536365,85123A
1,536365,85123A
2,536365,71053
3,536365,71053
4,536365,84406B


In [ ]:
# Create Co-purchases matrix
pr_df['count'] = 1 # match labelled as 1 

Counts = pr_df.pivot_table(
    values='count',
    index='InvoiceNo',
    columns='StockCode',
    aggfunc='max', # existence of a row in pr_df yields max(count) = 1.
    fill_value=0 # 0 for no matches
    )

stockcodelist = [str(code) for code in Counts.columns]
C = Counts.values

CoOrderMatrix = C.T @ C # create co-order matrix from invoice order matrix


In [4]:
# Create Jaccard similarity matrix

product_counts = CoOrderMatrix.diagonal()

n = len(stockcodelist)
Oc = np.zeros(shape=(n,n))

for i in range(0,n):
    for j in range(0,n):
        Oc[i,j] = product_counts[i]+product_counts[j]

Oc = Oc - np.diag(product_counts)

CoOrderJaccard = np.divide(CoOrderMatrix,Oc) 



In [ ]:
# Helper function to melt and get top n co-purchased products
def melt_and_top_n(df, value_col, n=5):
    return (
        df
        .melt(
            id_vars='StockID1',
            var_name='StockID2',
            value_name=value_col
        )
        .query("StockID1 != StockID2")   # remove self-pairs
        .sort_values(['StockID1', value_col], ascending=[True, False])
        .groupby('StockID1', group_keys=False)
        .head(n)
    )


# Co-purchase counts
CoPurchases_df = pd.DataFrame(
    data=CoOrderMatrix,
    index=stockcodelist,
    columns=stockcodelist
).reset_index(names='StockID1')


top5_copurchases = melt_and_top_n(
    CoPurchases_df,
    value_col='Copurchases',
    n=5
)


# Jaccard Similarity
CoPurchasesJaccard_df = pd.DataFrame(
    data=CoOrderJaccard,
    index=stockcodelist,
    columns=stockcodelist
).reset_index(names='StockID1')


top5_jaccard = melt_and_top_n(
    CoPurchasesJaccard_df,
    value_col='Jaccard Similarity',
    n=5
)

In [6]:
# Save as CSV file
OUT_DIR = '../data/analysis'
top5_copurchases.to_parquet(OUT_DIR + '/copurchases.parquet', index=False)
top5_jaccard.to_parquet(OUT_DIR+'/copurchases_jaccard.parquet', index=False)